In [ ]:
from naive_adc_circuit import naive_circuit
from hardware_aware_circuit import ha_circuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import generate_preset_pass_manager
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import mapomatic as mm
plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 200

In [2]:
service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    instance='crn:v1:bluemix:public:quantum-computing:eu-de:a/cb804b30dfcb48b890393bfd6e41e9c2:a59c3379-0cf7-47fe-adf1-553bf24e0c35::'
)
backend = service.backend("ibm_basquecountry")

In [9]:
n = 5
J = [1/4]*(n-1)
gamma = [J[0]/4]*n
tlist = np.linspace(0, 25, 100)
excited = ["0"]
k = 1
dissipation = True

qc = naive_circuit(J, gamma, n, excited, k, dissipation)
naive_pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
isa_qc = naive_pm.run(qc)

small_qc = mm.deflate_circuit(isa_qc)
layouts = mm.matching_layouts(small_qc, backend)
scores = mm.evaluate_layouts(small_qc, layouts, backend)

print(f"The best layout in {backend} is: {scores[0][0]}")

The best layout in <IBMBackend('ibm_basquecountry')> is: [40, 41, 36, 21, 42, 56, 43, 44, 45, 37, 46]


In [10]:
mm.best_overall_layout(small_qc, service.backends(), successors=True)

[([40, 41, 36, 21, 42, 56, 43, 44, 45, 37, 46],
  'ibm_basquecountry',
  0.08934631023474737)]

The score is the infidelity (1 - fid) so the smaller the less error the layout has and the better it is

In [ ]:
qc, init_layout = ha_circuit(J, gamma, n, excited, k, backend, dissipation)
pm = generate_preset_pass_manager(optimization_level=3, backend=backend, routing_method="none", seed_transpiler=123)
isa_qc = naive_pm.run(qc)

small_qc = mm.deflate_circuit(isa_qc)
layouts = mm.matching_layouts(small_qc, backend, strict_direction=False) # Have to remove direction to allow for more flexible layouts
scores = mm.evaluate_layouts(small_qc, layouts, backend)

print(f"The best layout in {backend.name} by Map'omatic is: {scores[0][0]} with score {scores[0][1]}")

The best layout in ibm_basquecountry by Map'omatic is: [42, 43, 56, 41, 36, 21, 22] with score 0.08389444982900351


In [78]:
exists = any(set(best_chain) == set(lst) for lst in layouts)
exists

True

In [79]:
best_chain = list(init_layout.get_physical_bits().keys())
match = next((lst for lst in layouts if set(best_chain) == set(lst)), None)
score = mm.evaluate_layouts(small_qc, match, backend)
print(f"The general_qlist is: {best_chain} with score {score[0][1]}")

The general_qlist is: [56, 63, 62, 61, 76, 81, 82] with score 0.09606970494349876


In [80]:
default_trans = list(isa_qc.layout.initial_layout.get_physical_bits().keys())[0:(n + np.floor_divide(n,2))]
match = next((lst for lst in layouts if set(default_trans) == set(lst)), None)
score = mm.evaluate_layouts(small_qc, match, backend)
print(f"The default layout chosen by the transpiler is: {default_trans} with score {score[0][1]}")

The default layout chosen by the transpiler is: [22, 21, 36, 41, 43, 23, 42] with score 0.09385465712185315


In [ ]:
mm.evaluate_layouts(small_qc, layouts, backend)